# DAIL-SQL Baseline — Evaluation (Kaggle)

Notebook ini menjalankan **DAIL-SQL** (Gao et al., 2023) sebagai **baseline skripsi** pada Spider dev set, lalu mengevaluasinya dengan **lapisan metrik yang sama** seperti `eval_pipeline.ipynb` (GraphRAG 1-Stage dan 2-Stage). Logikanya ada di `src/retrieval/dail.py` dan `src/experiments/dail_steps.py`; kode DAIL asli ada di `external/dail_sql/` dan **tidak diubah**.

## Apa yang diambil dari DAIL-SQL, apa yang diganti

| Bagian | DAIL-SQL asli (`run_dail_sql.sh`) | Di skripsi ini |
|---|---|---|
| Representasi schema | `SQL`: CREATE TABLE **seluruh DB** (dari `sqlite_master`), tanpa schema linking | **sama** |
| Organisasi contoh | `QA`: pasangan pertanyaan–SQL, tanpa schema contoh | **sama** |
| Seleksi contoh | pass 1 `EUCDISQUESTIONMASK` → SQL awal; pass 2 `EUCDISMASKPRESKLSIMTHR` (DAIL Selection: pertanyaan ter-mask + kemiripan skeleton SQL awal ≥ 0.85) | **sama** |
| Embedding seleksi | `all-mpnet-base-v2` (CPU), jarak Euclidean | **sama** |
| k contoh, pool contoh | 9, `train_spider` + `train_others` | **sama** |
| Cleaner SQL | `ask_llm.py` (rapikan spasi, potong di `/*`, awali `SELECT`) | **sama** |
| LLM (kedua pass) | GPT-4 | **Qwen2.5-Coder** sesuai `MODEL`, 4-bit, greedy — dimuat dan dipanggil persis seperti 1-Stage/2-Stage |
| Batas output | 200 token | `cfg.max_new_tokens` (256, sama dengan 1-Stage/2-Stage), stop di `;` (mode completion DAIL) |
| Self-consistency | opsional | tidak dipakai (greedy) |
| Data | Spider dev penuh | sampel stratified yang **sama** dengan `eval_pipeline.ipynb` (`SAMPLE_RATIO`, seed) |

## Evaluasi: sama dengan `eval_pipeline.ipynb`

| Bagian | Fungsi yang dipakai |
|---|---|
| ESM, EX, CM, kolom gold | `SqlEvaluator` (notebook_steps.py) |
| Satu catatan per pertanyaan | `_log_entry()` (notebook_steps.py) → `raw_logs/dail_log.json`, skema sama dengan `baseline_log.json`/`graphrag_log.json` |
| SLA, ESM, EX, CM, Token, QVT | fungsi agregasi `metrics_summary.py` |
| TEP | rumus yang sama, dengan **DAIL-SQL sebagai B** (pembanding); hanya kalau raw_logs 1-Stage/2-Stage model yang sama sudah ada |
| Token | T = T_in + α × T_out, dengan T_in dan T_out = **pass 1 + pass 2** (DAIL memanggil LLM dua kali) |
| Latensi | tahap yang sama dengan pipeline kita (`build_ddl`, `few_shot`, `build_prompt`, `prompt_tokenize`, `llm_generate`, `sql_clean`, `end_to_end`), dijumlah untuk kedua pass, ditambah total per pass |
| Evaluator resmi | `external/spider_eval/evaluation.py`, fungsi yang sama |

## Alur notebook

| Bagian | Isi |
|---|---|
| 0 | Preprocessing DAIL (sekali saja, biasanya di luar notebook ini) |
| 1-3 | Setup, import, **pilih `MODEL`** |
| 4-7 | Baca data, muat LLM, siapkan DAIL, siapkan evaluator |
| 8 | Contoh satu pertanyaan (boleh dilewati) |
| 9-10 | Jalankan DAIL-SQL, amankan prediksi dan `raw_logs` |
| 11 | Hasil metrik (dan perbandingan dengan 1-Stage/2-Stage kalau raw_logs-nya ada) |
| 12 | Evaluator resmi Spider |
| 13 | Simpan semua hasil |

> **Supaya perbandingannya berpasangan:** jalankan `eval_pipeline.ipynb` dan notebook ini dengan `MODEL` dan `SAMPLE_RATIO` yang sama. Hasil DAIL ditulis ke file `dail_*` di folder model yang sama, jadi tidak menimpa hasil 1-Stage/2-Stage.

## 0. Preprocessing DAIL (sekali saja)

DAIL me-mask pertanyaan memakai hasil schema linking ala RAT-SQL (`external/dail_sql/data_preprocess.py`). Hasilnya cukup dibuat **sekali** lalu dipakai semua run dan semua model:

```
data/dail/
├── train_spider_and_others.json
└── enc/
    ├── test_schema-linking.jsonl    (1034 baris, = dev.json)
    └── train_schema-linking.jsonl   (8659 baris, = train_spider + train_others)
```

Preprocessing butuh **Java + Stanford CoreNLP 2018-10-05**, `stanford-corenlp`, `torchtext` (GloVe 42B, ~5 GB), `bpemb`, `attrs`, `nltk`. `torchtext` sering bentrok dengan versi torch di Kaggle, jadi sebaiknya dijalankan di **environment terpisah** (mis. conda Python 3.8 sesuai README DAIL, atau notebook Kaggle CPU tersendiri), lalu folder `data/dail/` di-upload sebagai **Kaggle Dataset** dan diarahkan lewat `DAIL_DATA_DIR` di Bagian 3.

```bash
# di environment preprocessing
pip install -r external/dail_sql/requirements.txt
python -c "import nltk; [nltk.download(p) for p in ('punkt', 'stopwords')]"
# unduh http://nlp.stanford.edu/software/stanford-corenlp-full-2018-10-05.zip lalu ekstrak ke
# external/dail_sql/third_party/  (atau set CORENLP_HOME)
python src/experiments/dail_preprocess.py --spider-dir <folder spider_data> --out-dir data/dail
```

Script itu memanggil fungsi DAIL asli dan mengecek jumlah baris linking sama dengan jumlah pertanyaan. Sel setelah setup (Bagian 1) hanya menjalankannya kalau `RUN_PREPROCESS = True`.

In [ ]:
# ============================================================
# 1. SETUP — clone repo, install dependency, cek evaluator Spider
# Sel ini aman dijalankan ulang (semua langkah dicek dulu).
# ============================================================
# os untuk pindah folder, subprocess untuk menjalankan perintah terminal
import os, subprocess
# Path untuk menulis lokasi file dengan aman di semua sistem operasi
from pathlib import Path

# Alamat repo yang di-clone saat notebook jalan di Kaggle
REPO_URL = "https://github.com/william-darma-wijaya/thesis-2803.git"
# Branch yang berisi DAIL-SQL
REPO_BRANCH = "feat/dail-sql"

# Fungsi pembantu untuk menjalankan satu perintah terminal
def sh(cmd: str) -> None:
    # Tampilkan perintahnya supaya terlihat apa yang sedang dijalankan
    print(f"$ {cmd}")
    # Jalankan perintah; check=True membuat notebook berhenti kalau perintahnya gagal
    subprocess.run(cmd, shell=True, check=True)

# Fungsi pembantu untuk mengecek apakah sebuah folder adalah root repo ini
def is_repo_root(p: Path) -> bool:
    # Cek file milik repo ini (Kaggle punya /kaggle/src sendiri)
    return (p / "src" / "core" / "config.py").is_file()

# cwd HARUS root repo: path relatif (outputs/, data/, external/) dipakai di mana-mana
if not is_repo_root(Path.cwd()):
    # Notebook dijalankan dari notebooks/
    if is_repo_root(Path.cwd().parent):
        # Naik satu folder ke root repo
        os.chdir("..")
    # Kaggle: repo belum ada atau ada di subfolder
    else:
        # Repo belum pernah di-clone
        if not is_repo_root(Path("thesis-2803").resolve()):
            # Clone branch DAIL
            sh(f"git clone -b {REPO_BRANCH} {REPO_URL}")
        # Repo sudah ada dari run sebelumnya
        else:
            # Ambil versi terbaru dari branch yang dipilih
            sh(f"git -C thesis-2803 fetch origin {REPO_BRANCH} && git -C thesis-2803 checkout {REPO_BRANCH} && git -C thesis-2803 pull origin {REPO_BRANCH}")
        # Masuk ke folder repo
        os.chdir("thesis-2803")
# Simpan lokasi root repo
REPO_ROOT = Path.cwd()
# Hentikan notebook kalau ternyata masih bukan root repo
assert is_repo_root(REPO_ROOT), f"{REPO_ROOT} bukan root repo thesis-2803"
# Tampilkan lokasi repo dan branch
print("Repo root:", REPO_ROOT)
print("Branch   :", subprocess.run("git rev-parse --abbrev-ref HEAD", shell=True, capture_output=True, text=True).stdout.strip())

# Dependency eval_pipeline + dependency DAIL saat run: sql_metadata untuk skeleton SQL (harus 2.x: API 3.x
# berubah dan membuat sql2skeleton() DAIL crash), sqlparse untuk post_process.py DAIL
sh("pip install -q -U 'bitsandbytes>=0.46.1' sentence-transformers transformers nltk 'sql_metadata<3' sqlparse")

# Evaluator resmi Spider dan kode DAIL harus ada di repo
for path in ("external/spider_eval/evaluation.py", "external/spider_eval/process_sql.py", "external/dail_sql/prompt/prompt_builder.py"):
    # Hentikan notebook kalau filenya tidak ada
    assert (REPO_ROOT / path).exists(), f"{path} tidak ada di repo"

# nltk: punkt untuk parser Spider; stopwords dibaca kode DAIL saat di-import
import nltk
# Unduh paket yang dibutuhkan
for pkg in ("punkt", "punkt_tab", "stopwords"):
    # Coba unduh paketnya
    try:
        nltk.download(pkg, quiet=True)
    # Kalau gagal, beri peringatan tanpa menghentikan notebook
    except Exception as e:
        print(f"[WARN] nltk.download({pkg}) gagal: {e}")
# Tanda setup selesai
print("Setup selesai.")

In [ ]:
# True hanya kalau environment ini sudah punya Java, CoreNLP, dan torchtext; biasanya False
RUN_PREPROCESS = False
# Jalankan preprocessing kalau diminta (repo sudah di-clone oleh sel di atas)
if RUN_PREPROCESS:
    # Panggil script preprocessing dengan dataset Spider di Kaggle; hasilnya ke data/dail/
    sh("python src/experiments/dail_preprocess.py --spider-dir /kaggle/input/datasets/alrette/spiderdataset/spider_data --out-dir data/dail")

## 2. Import

Langkah notebook ada di `src/experiments/dail_steps.py` (dipanggil lewat `dsteps`), yang memakai ulang langkah `eval_pipeline` di `src/experiments/notebook_steps.py` (`steps`).

In [ ]:
# sys untuk mendaftarkan lokasi kode
import sys
# Daftarkan root repo supaya modul src bisa di-import
sys.path.insert(0, str(REPO_ROOT))

# Konfigurasi utama pipeline
from src.core.config import PipelineConfig
# Folder hasil per model, sama dengan eval_pipeline
from src.experiments.pipeline import model_run_dirs
# Fungsi menampilkan tabel di notebook
from src.experiments.result_tables import show
# Langkah eval_pipeline yang dipakai ulang
from src.experiments import notebook_steps as steps
# Langkah khusus DAIL-SQL
from src.experiments import dail_steps as dsteps

# Pastikan semua import berhasil
print("Imports OK")

## 3. Konfigurasi

**Yang diganti tiap orang hanya `MODEL`**, sama dengan `eval_pipeline.ipynb`. Pakai `SAMPLE_RATIO` yang sama dengan run 1-Stage/2-Stage supaya pertanyaannya sama.

| Pengaturan | Arti |
|---|---|
| `SAMPLE_RATIO` | `0.05` untuk percobaan cepat, `1.0` untuk run final |
| `DAIL_DATA_DIR` | Folder hasil Bagian 0 (`train_spider_and_others.json` + `enc/`). Di Kaggle: folder Kaggle Dataset hasil upload |

Pengaturan DAIL lain (k = 9, selector, representasi) ada di `src/core/config.py` (`dail_*`) dan sengaja tidak diubah: DAIL-SQL dijalankan apa adanya.

In [ ]:
# ===== BAGIAN YANG DIGANTI TIAP ORANG: hanya MODEL =====
# Model LLM yang dijalankan: "3B", "7B", atau "14B"
MODEL = "7B"
# Porsi dev set yang dipakai: 0.05 = percobaan cepat, 1.0 = run final skripsi
SAMPLE_RATIO = 0.05
# Lokasi hasil preprocessing DAIL; di Kaggle ganti dengan folder dataset hasil upload
DAIL_DATA_DIR = Path("data/dail")

# Konfigurasi default (pengaturan DAIL ada di field dail_*)
cfg = PipelineConfig()
# Ubah nama pendek ("7B") menjadi nama lengkap model
cfg.llm_model = cfg.resolve_llm_model(MODEL)
# Arahkan ke hasil preprocessing DAIL
cfg.dail_data_dir = DAIL_DATA_DIR
# Hentikan notebook kalau dataset Spider tidak ditemukan
assert cfg.data_path.exists(), f"Data Spider tidak ditemukan di {cfg.data_path}. Di Kaggle, tambahkan lewat 'Add Input'."

# True kalau ini run penuh
IS_FULL = SAMPLE_RATIO >= 1.0
# Folder hasil, raw_logs, dan QVT model ini (sama dengan eval_pipeline)
ARTIFACT_DIR, RAW_LOGS_DIR, QVT_DIR = model_run_dirs(cfg.llm_model, out_root=Path("outputs") if IS_FULL else Path("outputs/trial"))
# Run percobaan menyimpan raw_logs di folder trial
if not IS_FULL:
    RAW_LOGS_DIR = ARTIFACT_DIR / "raw_logs"
# Buat folder prediksi, tabel, dan raw_logs
for folder in (ARTIFACT_DIR / "predictions", ARTIFACT_DIR / "tables", RAW_LOGS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# Kunci semua sumber acak
steps.set_seed(cfg.seed)
# Tampilkan konfigurasi run
dsteps.show_run_config(cfg, SAMPLE_RATIO, ARTIFACT_DIR, RAW_LOGS_DIR)

## 4. Baca Data

Fungsi yang sama dengan `eval_pipeline.ipynb`: sampel per tingkat kesulitan dengan seed tetap, jadi pertanyaannya sama persis dengan run 1-Stage/2-Stage.

In [ ]:
# Baca pertanyaan dev set; run percobaan hanya mengambil sebagian
questions = steps.load_dev_questions(cfg, SAMPLE_RATIO)
# Tulis gold SQL sesuai sampel, untuk evaluator resmi di Bagian 12
gold_path = steps.write_gold_file(questions, ARTIFACT_DIR)
# Tampilkan contoh pertanyaan pertama
print("Contoh:", questions[0]["question"])

## 5. Muat LLM

LLM yang sama dengan 1-Stage/2-Stage (4-bit, greedy). DAIL tidak memakai BGE-M3. Pastikan Accelerator = GPU.

In [ ]:
# Muat LLM dan tokenizer sesuai MODEL
llm, tokenizer = dsteps.load_llm(cfg)

## 6. Siapkan DAIL-SQL

Membaca dev/train beserta hasil schema linking, lalu membuat dua index contoh (pass 1 dan pass 2) dengan `all-mpnet-base-v2` di CPU, persis seperti `generate_question.py` DAIL. Langkah ini makan beberapa menit dan dicatat sebagai waktu sekali-jalan (`dail_setup`).

In [ ]:
# Cek hasil preprocessing lalu siapkan DAIL
dail = dsteps.load_dail(cfg)

## 7. Siapkan Evaluator

Evaluator yang sama dengan `eval_pipeline.ipynb` (`src/metrics/esm_ex_cm.py`).

In [ ]:
# Alat untuk menghitung ESM, EX, CM, dan kolom gold
evaluator = steps.SqlEvaluator(cfg)
# Satukan semua alat
tools = dsteps.DailTools(cfg, dail, llm, tokenizer, evaluator, dsteps.columns_per_table(evaluator))

## 8. Contoh Satu Pertanyaan

Menampilkan prompt pass 1, SQL awal dan skeleton-nya, prompt pass 2 (contoh dipilih ulang dengan DAIL Selection), dan SQL final. Ini hanya contoh dan **tidak masuk hasil akhir**, jadi boleh dilewati. `SHOW_PROMPTS = False` menyembunyikan prompt yang panjang.

In [ ]:
# Posisi pertanyaan yang ingin dilihat
WALK_IDX = 0
# Tampilkan prompt lengkap atau tidak
SHOW_PROMPTS = True
# Jalankan kedua pass untuk pertanyaan itu
example = dsteps.walk_through(tools, questions[WALK_IDX], show_prompts=SHOW_PROMPTS)

## 9. Jalankan DAIL-SQL

Untuk tiap pertanyaan: pass 1 → SQL awal → skeleton → pass 2 → SQL final, lalu evaluasi dan simpan satu catatan. Kalau generate gagal, SQL diganti `SELECT 1` dan run tetap lanjut (sama dengan `eval_pipeline`). Untuk run final jumlah error sebaiknya 0.

In [ ]:
# Jalankan DAIL-SQL untuk semua pertanyaan
results = dsteps.run_all_questions(tools, questions)
# Tampilkan jumlah pertanyaan dan error
dsteps.show_loop_summary(results)

## 10. Amankan Prediksi dan raw_logs

Disimpan sekarang supaya hasil generate yang lama tidak hilang kalau langkah berikutnya gagal:

| File | Isi |
|---|---|
| `predictions/dail_predictions.txt` | SQL final, satu per baris |
| `predictions/dail_prompts.jsonl` | Prompt dan output mentah kedua pass (audit) |
| `raw_logs/dail_log.json` | Catatan per pertanyaan, skema sama dengan 1-Stage/2-Stage, ditambah field `dail` (SQL awal, skeleton, token per pass) |

In [ ]:
# Simpan prediksi, prompt, dan raw_logs
dsteps.save_generation_results(results, ARTIFACT_DIR, RAW_LOGS_DIR)

## 11. Hasil Metrik

Dihitung dari `raw_logs` dengan fungsi agregasi yang sama dengan `metrics_results.json` 1-Stage/2-Stage. Kalau `baseline_log.json` dan `graphrag_log.json` model ini sudah ada **dengan pertanyaan yang sama**, kolom 1-stage dan 2-stage ikut ditampilkan dan TEP dihitung dengan DAIL-SQL sebagai B. QVT hanya terisi pada run penuh. Hasil di bagian ini belum disimpan.

In [ ]:
# Hitung metrik DAIL (dan 1-Stage/2-Stage kalau raw_logs-nya sebanding)
metrics = dsteps.compute_metrics(cfg, RAW_LOGS_DIR, SAMPLE_RATIO, QVT_DIR)
# raw_logs semua kondisi yang dibandingkan
logs_by_cond = dsteps.condition_logs(RAW_LOGS_DIR, metrics)
# Semua tabel hasil (yang sama akan ditulis ke Excel di Bagian 13)
groups = dsteps.table_groups(cfg, metrics, logs_by_cond)
# Tabel ringkasan keenam metrik
dsteps.show_group(groups, "Summary")

### 11.1 Per Tingkat Kesulitan

In [ ]:
# EX, ESM, dan token per tingkat kesulitan
dsteps.show_group(groups, "By Difficulty")

### 11.2 SLA

DAIL mengirim seluruh schema, jadi pada pertanyaan dengan gold schema tidak kosong recall = 100% dan precision = proporsi kolom gold di antara semua kolom DB. Populasi *All queries* juga memuat pertanyaan dengan gold schema kosong (mis. `count(*)`), sama seperti 1-Stage/2-Stage.

In [ ]:
# SLA per kondisi
dsteps.show_group(groups, "M1 SLA")

### 11.3 Component Match (CM)

In [ ]:
# CM per klausa
dsteps.show_group(groups, "M4 CM")

### 11.4 Token Consumption dan TEP

T_in dan T_out DAIL adalah jumlah kedua pass; rinciannya per pass ada di tabel ketiga.

In [ ]:
# Token, TEP, dan rincian per pass
dsteps.show_group(groups, "M5 Token TEP")

### 11.5 QVT

In [ ]:
# QVT
dsteps.show_group(groups, "M6 QVT")

### 11.6 Tabel Pendukung dan Latensi

Latensi adalah lampiran (bukan salah satu dari 6 metrik). Pertanyaan pertama tidak ikut statistik (pemanasan model).

In [ ]:
# Pengecekan prediksi
dsteps.show_group(groups, "SQL Diagnostics")
# Latensi per tahap, per tingkat kesulitan, dan ukuran turunan
dsteps.show_group(groups, "Latency")

## 12. Pembanding: Evaluator Resmi Spider

Fungsi yang sama dengan Bagian 14 `eval_pipeline.ipynb`. Hasilnya belum disimpan.

In [ ]:
# Jalankan evaluator resmi Spider untuk prediksi DAIL
spider_scores = dsteps.run_spider_check(cfg, ARTIFACT_DIR, gold_path)

## 13. Simpan Semua Hasil

Semua file DAIL masuk folder terpisah `tables/dail/`, jadi `metrics_results.json`, `latency_run.json`, dan `hasil_skripsi.xlsx` milik 1-Stage/2-Stage tidak tertimpa.

| File | Isi |
|---|---|
| `tables/dail/dail_metrics_results.json` | Metrik semua kondisi yang dibandingkan + pengaturan DAIL |
| `tables/dail/hasil_dail.xlsx` | Semua tabel di atas |
| `tables/dail/spider_official_eval.csv` | Skor evaluator resmi |
| `tables/dail/latency_run.json` | Waktu sekali-jalan, GPU, puncak VRAM |

Di Kaggle juga dibuat `hasil_dail_<MODEL>.xlsx` dan `hasil_dail_<MODEL>.zip` di `/kaggle/working/`.

In [ ]:
# Simpan metrik, skor resmi, latensi, dan Excel
xlsx_path = dsteps.save_all_results(cfg, MODEL, metrics, logs_by_cond, spider_scores, ARTIFACT_DIR, RAW_LOGS_DIR)

## Catatan Desain yang Perlu Disebut di Metodologi

- **DAIL-SQL direproduksi apa adanya, hanya LLM yang diganti.** Konfigurasi DAIL (k = 9, representasi `SQL`, organisasi `QA`, DAIL Selection) adalah konfigurasi terbaik dari paper DAIL sendiri dan tidak dituning ulang. Konfigurasi itu dicari untuk GPT-4, jadi belum tentu optimal untuk Qwen (keterbatasan).
- **SQL awal (pass 1) juga dari Qwen yang sama**, bukan GPT-4 atau Graphix seperti di paper, supaya tidak ada model lain yang membantu DAIL.
- **Token = pass 1 + pass 2.** DAIL memanggil LLM dua kali per pertanyaan, dan keduanya dihitung.
- **Prompt mentah tanpa chat template**, sama dengan 1-Stage/2-Stage. Prompt DAIL diakhiri `SELECT `, jadi Qwen dipakai dalam mode completion; stop di `;` mengikuti mode completion DAIL (`llm/chatgpt.py`).
- **Schema DAIL diambil dari file `.sqlite`** (`sqlite_master`), sedangkan SLA memakai daftar kolom `tables.json` (sama dengan 1-Stage/2-Stage). Isinya sama untuk Spider; yang berbeda hanya teks DDL.
- **Pool contoh DAIL = `train_spider` + `train_others`**, sedangkan 1-Stage/2-Stage memakai `train_spider`. Keduanya tetap retrieval dari train set (bukan kebocoran dev).
- **Embedding seleksi contoh DAIL = `all-mpnet-base-v2`**, bukan BGE-M3, karena bagian dari arsitektur DAIL.